# Goal-Conditioned Analog AI Sizing (V12) - Kaggle Version
V12 Features (Reward Scaling Fix):
- Scaled reward (-cost/100) instead of clipped — full gradient preserved
- No early termination (V11 fix preserved)
- Smooth normalized cost function (V10 fix preserved)
- Rich 15D Observation + Direct Parameter Output
- Training: 1,000,000 steps

## Data Sources
1. **Code Dataset:** analog-ai-code-v12
2. **LUT Source:** ota-rl-model-v9/v10 kernel output

In [ ]:
!pip install stable-baselines3[extra] gymnasium numpy scipy tensorboard gdown

In [ ]:
import os
import sys

WORKING_DIR = '/kaggle/working/'
print("Finding LUTs from all Kaggle inputs...")
nch_path = None
pch_path = None

for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        fp = os.path.join(root, file)
        if file == 'TSMC_fast_65nm_nch.pkl' and os.path.getsize(fp) > 1000000:
            nch_path = fp
        if file == 'TSMC_fast_65nm_pch.pkl' and os.path.getsize(fp) > 1000000:
            pch_path = fp

if nch_path and pch_path:
    print(f"Found NCH: {nch_path}")
    print(f"Found PCH: {pch_path}")
else:
    print("ERROR: Could not find valid LUTs in /kaggle/input/")
    print("Listing all /kaggle/input/ contents:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        for f in files:
            print(f"  {os.path.join(root, f)} ({os.path.getsize(os.path.join(root, f))} bytes)")

In [ ]:
import os
import sys
import gdown
import zipfile

# 1. Find the project root from Kaggle dataset
dataset_path = '/kaggle/input'
project_root = None

for root, dirs, files in os.walk(dataset_path):
    if 'core' in dirs and 'circuits' in dirs and 'v12' in root.lower():
        project_root = root
        break

if project_root is None:
    for root, dirs, files in os.walk(dataset_path):
        if 'core' in dirs and 'circuits' in dirs:
            project_root = root
            break

if project_root:
    print(f'Found project root at: {project_root}')
    sys.path.append(project_root)
else:
    print('Error: Could not find project root in dataset.')

In [ ]:
from tech_luts.lut_utils import LUT
from core.device_model import DeviceModel
from circuits.ota5t import OTA5T
from optimizer.rl_environment import OTA5tGymEnv

print("Loading LUTs into memory...")

nch = LUT(nch_path)
pch = LUT(pch_path)

dm = DeviceModel(nch, pch)
ota = OTA5T(dm, vdd=1.2)
print("Physics Engine Ready!")

In [ ]:
bounds = [
    (60e-9, 1.5e-6),  # L1
    (5.0, 25.0),      # gmid1
    (60e-9, 1.5e-6),  # L3
    (5.0, 25.0),      # gmid3
    (10e-6, 500e-6)   # Itail
]

env = OTA5tGymEnv(ota, bounds=bounds, max_steps=200)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

checkpoint_callback = CheckpointCallback(
    save_freq=100000, 
    save_path=os.path.join(WORKING_DIR, 'checkpoints'),
    name_prefix='kaggle_ppo_model_v12'
)

In [ ]:
tb_log_dir = os.path.join(WORKING_DIR, 'ppo_ota_tensorboard')

# V12: Larger network (256x256) for goal-conditioned policy.
# Scaled reward preserves full gradient for fine-grained target conditioning.
policy_kwargs = dict(net_arch=[256, 256])
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0001, batch_size=512, n_steps=2048, policy_kwargs=policy_kwargs, tensorboard_log=tb_log_dir)

print("Starting 1,000,000 Steps Training on Kaggle (V12 - Reward Scaling Fix)...")
model.learn(total_timesteps=1000000, callback=checkpoint_callback)

model_save_path = 'universal_ppo_agent_65nm_v12'
model.save(model_save_path)
print(f'Model saved to {model_save_path}.zip')